In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE

pd.set_option('display.max_columns', 50)

print("✅ Bibliothèques importées !")

✅ Bibliothèques importées !


In [2]:
# Chargement du dataset original
df = pd.read_csv('../data/raw/diabetic_data.csv')

# On travaille sur une copie pour ne jamais toucher les données brutes
df_clean = df.copy()

print(f"✅ Dataset chargé : {df_clean.shape[0]} lignes × {df_clean.shape[1]} colonnes")

✅ Dataset chargé : 101766 lignes × 50 colonnes


In [3]:
print("="*50)
print("ÉTAPE 1 : SUPPRESSION DES COLONNES INUTILES")
print("="*50)

# Colonnes avec trop de valeurs manquantes (>40%)
cols_to_drop = ['weight', 'max_glu_serum', 'A1Cresult', 'payer_code']

# Colonnes non pertinentes pour la prédiction
cols_to_drop += ['encounter_id', 'patient_nbr']

df_clean = df_clean.drop(columns=cols_to_drop)

print(f"Colonnes supprimées : {cols_to_drop}")
print(f"\n✅ Nouvelles dimensions : {df_clean.shape[0]} lignes × {df_clean.shape[1]} colonnes")

ÉTAPE 1 : SUPPRESSION DES COLONNES INUTILES
Colonnes supprimées : ['weight', 'max_glu_serum', 'A1Cresult', 'payer_code', 'encounter_id', 'patient_nbr']

✅ Nouvelles dimensions : 101766 lignes × 44 colonnes


In [4]:
print("="*50)
print("ÉTAPE 2 : TRAITEMENT DES VALEURS MANQUANTES")
print("="*50)

# Remplacer les '?' par NaN
df_clean = df_clean.replace('?', np.nan)

# Remplacer les NaN de medical_specialty par 'Unknown'
df_clean['medical_specialty'] = df_clean['medical_specialty'].fillna('Unknown')

# Remplacer les NaN de race par 'Unknown'
df_clean['race'] = df_clean['race'].fillna('Unknown')

# Remplacer les NaN de diag_1, diag_2, diag_3 par 'Unknown'
df_clean[['diag_1', 'diag_2', 'diag_3']] = df_clean[['diag_1', 'diag_2', 'diag_3']].fillna('Unknown')

# Vérification
remaining = df_clean.isnull().sum().sum()
print(f"✅ Valeurs manquantes restantes : {remaining}")

ÉTAPE 2 : TRAITEMENT DES VALEURS MANQUANTES
✅ Valeurs manquantes restantes : 0


In [5]:
print("="*50)
print("ÉTAPE 3 : ENCODAGE DE LA VARIABLE CIBLE")
print("="*50)

# Transformer readmitted en variable binaire
# <30 = 1 (réadmission rapide, cas critique)
# >30 et NO = 0
df_clean['readmitted'] = df_clean['readmitted'].map({'<30': 1, '>30': 0, 'NO': 0})

print("Distribution de la variable cible encodée :")
print(df_clean['readmitted'].value_counts())
print(f"\nTaux de réadmission <30j : {df_clean['readmitted'].mean()*100:.2f}%")
print("✅ Variable cible encodée !")

ÉTAPE 3 : ENCODAGE DE LA VARIABLE CIBLE
Distribution de la variable cible encodée :
readmitted
0    90409
1    11357
Name: count, dtype: int64

Taux de réadmission <30j : 11.16%
✅ Variable cible encodée !


In [6]:
print("="*50)
print("ÉTAPE 4 : ENCODAGE DES VARIABLES CATÉGORIELLES")
print("="*50)

# Colonnes catégorielles à encoder
cat_cols = df_clean.select_dtypes(include='str').columns.tolist()  # ← correction ici
print(f"Colonnes à encoder ({len(cat_cols)}) : {cat_cols}")

# Label Encoding
le = LabelEncoder()
for col in cat_cols:
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))

print(f"\n✅ Encodage terminé !")
print(f"Dimensions finales : {df_clean.shape[0]} lignes × {df_clean.shape[1]} colonnes")
df_clean.head(3)

ÉTAPE 4 : ENCODAGE DES VARIABLES CATÉGORIELLES
Colonnes à encoder (32) : ['race', 'gender', 'age', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'acetohexamide', 'glipizide', 'glyburide', 'tolbutamide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'troglitazone', 'tolazamide', 'examide', 'citoglipton', 'insulin', 'glyburide-metformin', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed']

✅ Encodage terminé !
Dimensions finales : 101766 lignes × 44 colonnes


,race,gender,age,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2,0,0,6,25,1,1,37,41,0,1,0,0,0,124,709,747,1,1,1,1,1,1,0,1,1,0,1,1,1,1,0,0,0,0,1,1,0,0,0,0,1,0,0
1,2,0,1,1,1,7,3,71,59,0,18,0,0,0,143,79,121,9,1,1,1,1,1,0,1,1,0,1,1,1,1,0,0,0,0,3,1,0,0,0,0,0,1,0
2,0,0,2,1,1,7,2,71,11,5,13,2,0,1,454,78,767,6,1,1,1,1,1,0,2,1,0,1,1,1,1,0,0,0,0,1,1,0,0,0,0,1,1,0


In [7]:
print("="*50)
print("ÉTAPE 5 : SÉPARATION X/y + SMOTE")
print("="*50)

# Séparation features / cible
X = df_clean.drop(columns=['readmitted'])
y = df_clean['readmitted']

print(f"Avant SMOTE : {y.value_counts().to_dict()}")

# Application du SMOTE
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

print(f"Après SMOTE  : {pd.Series(y_resampled).value_counts().to_dict()}")
print(f"\n✅ Dataset équilibré : {X_resampled.shape[0]} lignes × {X_resampled.shape[1]} colonnes")

ÉTAPE 5 : SÉPARATION X/y + SMOTE
Avant SMOTE : {0: 90409, 1: 11357}
Après SMOTE  : {0: 90409, 1: 90409}

✅ Dataset équilibré : 180818 lignes × 43 colonnes


In [8]:
print("="*50)
print("ÉTAPE 6 : SAUVEGARDE DES DONNÉES TRAITÉES")
print("="*50)

import os
os.makedirs('../data/processed', exist_ok=True)

# Sauvegarder le dataset nettoyé (sans SMOTE) pour référence
df_clean.to_csv('../data/processed/diabetic_clean.csv', index=False)

# Sauvegarder le dataset rééquilibré pour le ML
X_resampled_df = pd.DataFrame(X_resampled, columns=X.columns)
X_resampled_df['readmitted'] = y_resampled
X_resampled_df.to_csv('../data/processed/diabetic_resampled.csv', index=False)

print("✅ Fichiers sauvegardés dans data/processed/ :")
print("   - diabetic_clean.csv")
print("   - diabetic_resampled.csv")

ÉTAPE 6 : SAUVEGARDE DES DONNÉES TRAITÉES
✅ Fichiers sauvegardés dans data/processed/ :
   - diabetic_clean.csv
   - diabetic_resampled.csv
